# 💻 Notebook do Aluno — Aula 02: Memória conversacional Buffer, Summary e TokenBuffer

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 02/14 — Módulo 1: LangChain Foundations**  
**⏱️ 1h40min**  
**🧠 3 tipos de memória**  
**🔁 Andaime 45%**  

---

## 🎯 Objetivo da aula

Tornar a chain da Aula 01 stateful — com memória persistida entre turnos — escolhendo o tipo certo para o domínio do grupo, e entender o trade-off de custo de tokens de cada abordagem.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-ollama langchain-classic -q

from langchain_ollama import ChatOllama
from langchain_classic.memory import (
    ConversationBufferMemory,
    ConversationSummaryMemory,
    ConversationTokenBufferMemory,
)
from langchain_classic.chains import ConversationChain
from langchain_core.prompts import PromptTemplate
import os
from google.colab import userdata
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
llm = ChatOllama(model="gpt-oss:120b")

# 👉 LACUNA 1: escreva o template com a persona do domínio do grupo
# Lembre: deve conter {history} e {input}
TEMPLATE = ___

prompt = PromptTemplate(input_variables=["history", "input"], template=TEMPLATE)

# 👉 LACUNA 2: escolha o tipo de memória para o domínio do grupo
# Justifique em comentário: por que este tipo e não os outros?
memoria = ___  # Buffer, Summary ou TokenBuffer

# 👉 LACUNA 3: monte a ConversationChain com llm, memory e prompt
chat = ConversationChain(___)

# 👉 LACUNA 4: teste com 5 turnos do domínio do grupo
for pergunta in [___, ___, ___, ___, ___]:
    print(f"R: {chat.predict(input=pergunta)}\n")

# Inspecionar o estado final da memória
print(memoria.load_memory_variables({}))

---

## ✍️ Suas anotações

Registre aqui as observações da prática (qualidade dos resultados, comparações e conclusões do grupo).

---

## 🏋️ Exercícios da Aula 02

Os quatro exercícios praticam a escolha e o ajuste de memória para o chatbot do domínio do grupo — radiografia dos 3 tipos, janela de tokens e histórico por session_id. Complete os andaimes, rode cada célula no Colab e registre a recomendação final de memória do grupo como entrega da aula.


### Exercício 1 — Radiografia das 3 memórias · ★★☆ · 10 min

*Individual · Colab*

Os mesmos 3 turnos em três `ConversationChain` idênticas, uma por tipo de memória — compare o que cada uma guarda.

1. Complete o template com a persona do domínio (obrigatório: `{history}` e `{input}`).
2. Complete o dict `memorias` — o llm que gera o resumo e o limite de tokens do TokenBuffer.
3. Rode e compare a radiografia das três: mensagens literais, resumo em texto ou janela fixa?
4. Em comentário, indique qual dos 3 tipos serve melhor ao domínio do grupo.

> **💡 Dica:** o trade-off central é fidelidade × custo — Buffer guarda tudo (fiel e caro), Summary comprime com um LLM (custo quase fixo, +1 chamada por turno), TokenBuffer mantém janela fixa descartando as antigas.


In [ ]:
# 👉 LACUNA: a mesma conversa nas 3 memórias — compare o que cada uma guarda
from langchain_classic.memory import (
    ConversationBufferMemory, ConversationSummaryMemory, ConversationTokenBufferMemory)
from langchain_classic.chains import ConversationChain
from langchain_core.prompts import PromptTemplate

# 👉 LACUNA 1: persona do domínio no template — deve conter {history} e {input}
TEMPLATE = """___
Histórico da conversa:
{history}
Usuário: {input}
Assistente:"""
prompt = PromptTemplate(input_variables=["history", "input"], template=TEMPLATE)

turnos = ["Meu nome é Ana e tenho 28 anos.",
          "Trabalho como engenheira de dados.",
          "Estou aprendendo LangChain."]

memorias = {
    "buffer":      ConversationBufferMemory(memory_key="history", return_messages=True),
    # 👉 LACUNA 2: complete o llm que gera o resumo
    "summary":     ConversationSummaryMemory(llm=___, memory_key="history", return_messages=True),
    # 👉 LACUNA 3: complete o limite de tokens da janela
    "tokenbuffer": ConversationTokenBufferMemory(llm=llm, max_token_limit=___,
                                                 memory_key="history", return_messages=True),
}

for nome, mem in memorias.items():
    chat_x = ConversationChain(llm=llm, memory=mem, prompt=prompt, verbose=False)
    for p in turnos:
        chat_x.predict(input=p)
    print(f"--- {nome} ---")
    # Buffer: mensagens literais | Summary: resumo em texto | TokenBuffer: janela fixa
    print(str(mem.load_memory_variables({})["history"])[:220], "...\n")
# 👉 LACUNA 4: em comentário — qual memória preserva o detalhe do turno 1? E qual custa mais tokens por turno?


### Exercício 2 — Diagnóstico: o histórico que não para de crescer · ★★☆ · 10 min

*Individual · Colab*

No teste do lab, o prompt do `ConversationBufferMemory` cresce a cada turno — troque por uma memória com janela fixa.

1. Complete os parâmetros do TokenBuffer: o llm que conta tokens e o limite de 800.
2. Rode os mesmos 5 turnos do lab e confirme que as respostas continuam coerentes.
3. Imprima `len(estado["history"])` — o número estagna perto do limite?
4. Em comentário: qual cenário do domínio justificaria `ConversationSummaryMemory` no lugar?

> **💡 Dica:** o parâmetro `llm=llm` no TokenBuffer não é opcional — é ele quem conta os tokens do histórico para aplicar a janela.


In [ ]:
# 👉 LACUNA: troque a memória do lab por TokenBuffer com limite fixo
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_classic.chains import ConversationChain

# 👉 LACUNA 1: complete os parâmetros (llm para contar tokens; limite 800)
memoria = ConversationTokenBufferMemory(
    llm=___,                # usado para contar tokens
    max_token_limit=___,
    memory_key="history",
    return_messages=True,
)

chat = ConversationChain(llm=llm, memory=memoria, verbose=False)

# 👉 LACUNA 2: rode os mesmos 5 turnos do lab
for pergunta in [___, ___, ___, ___, ___]:
    print(f"R: {chat.predict(input=pergunta)}\n")

# Radiografia: Buffer cresce sem limite; TokenBuffer estagna perto do limite
estado = memoria.load_memory_variables({})
print("Mensagens guardadas:", len(estado["history"]))
# 👉 LACUNA 3: comente — por que este tipo (e não Buffer/Summary) para o domínio?


### Exercício 3 — Buffer vs Summary: a mesma conversa, duas memórias · ★★☆ · 10 min

*Individual · Colab*

A MESMA sequência de 4 turnos em duas chains, uma com Buffer e outra com Summary — compare as radiografias.

1. Complete o llm que gera o resumo do `ConversationSummaryMemory`.
2. Complete o loop com a lista de perguntas e rode as duas chains.
3. Compare as radiografias no output: quem guarda mensagens literais e quem guarda texto resumido? Após 10 turnos, quem preserva o detalhe do turno 1?

> **💡 Dica:** `load_memory_variables({})` é a radiografia da memória — Buffer lista `HumanMessage`/`AIMessage`; Summary devolve uma string resumida pelo LLM.


In [ ]:
# 👉 LACUNA: mesma conversa, duas memórias — compare a radiografia
from langchain_classic.memory import ConversationBufferMemory, ConversationSummaryMemory
from langchain_classic.chains import ConversationChain

perguntas = ["Meu nome é Ana e tenho 28 anos.",
             "Trabalho como engenheira de dados.",
             "Estou aprendendo LangChain.",
             "Qual meu nome e profissão?"]

memoria_buffer  = ConversationBufferMemory(memory_key="history", return_messages=True)
# 👉 LACUNA 1: complete o llm usado para gerar o resumo
memoria_summary = ConversationSummaryMemory(llm=___, memory_key="history", return_messages=True)

for nome_memoria, m in [("buffer", memoria_buffer), ("summary", memoria_summary)]:
    chat_x = ConversationChain(llm=llm, memory=m)
    for p in ___:                       # 👉 LACUNA 2: itere as perguntas
        chat_x.predict(input=p)
    estado = m.load_memory_variables({})
    print(f"--- {nome_memoria} ---")
    print(estado["history"])            # buffer: mensagens literais | summary: resumo em texto
# 👉 LACUNA 3: em comentário — qual memória preserva o detalhe do turno 1 após 10 turnos?


### Exercício 4 — Memória por session_id na chain LCEL · ★★☆ · 10 min

*Individual · Colab*

O padrão moderno (LangChain 0.3+) envolve a chain da Aula 01 com histórico por sessão.

1. Envolva a chain LCEL base com `RunnableWithMessageHistory` — complete o primeiro argumento e o `input_messages_key`.
2. Invoque com dois `session_id` diferentes ("Meu nome é Ana." / "Meu nome é Bruno.") e pergunte o nome em cada sessão.
3. Confirme no `store` os 2 históricos independentes — e o que acontece sem o `config` no `.invoke()`?

> **💡 Dica:** o `config={"configurable": {"session_id": ...}}` é o endereço da conversa — sem ele o wrapper não sabe qual histórico carregar.


In [ ]:
# 👉 LACUNA: memória por session_id na chain LCEL da Aula 01
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_base = prompt | llm | StrOutputParser()   # chain LCEL da Aula 01

# 👉 LACUNA 1: envolva a chain com RunnableWithMessageHistory
chain_com_hist = RunnableWithMessageHistory(
    ___,
    get_session_history,
    input_messages_key="___",   # a variável do seu template (ex.: "pergunta")
)

# 👉 LACUNA 2: 2 invocações com session_ids diferentes — cada sessão lembra só a dela
chain_com_hist.invoke({"pergunta": "Meu nome é Ana."}, config=___)
chain_com_hist.invoke({"pergunta": "Meu nome é Bruno."}, config=___)

print(chain_com_hist.invoke({"pergunta": "Qual é o meu nome?"},
                            config={"configurable": {"session_id": ___}}))
print(len(store), "sessões no store")   # esperado: 2 históricos independentes


## 📚 Referências da aula

- Docs LangChain — Memory: ConversationBufferMemory, SummaryMemory, TokenBufferMemory. python.langchain.com/docs/modules/memory
- Docs LangChain — ConversationChain: combinar memória com prompt customizado. python.langchain.com/docs/modules/chains
- Docs LangChain — RunnableWithMessageHistory: abordagem moderna para memória em LCEL (0.3+). python.langchain.com/docs/expression_language/how_to/message_history
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 3: Memória em agentes — a relevância do histórico para tomada de decisão.
- Paper Brown, T. et al. — "Language Models are Few-Shot Learners." NeurIPS, 2020. Seção sobre context window e por que o tamanho do contexto impacta diretamente o custo e a qualidade. arxiv.org/abs/2005.14165
- Ebook Freed, A.; Jacobs, C.; Rózsa, E. — Effective Conversational AI. Manning, 2025. Cap. 9: Harnessing Context for an Adaptive Virtual Assistant Experience — a distinção entre session history e persistent user history que fundamenta esta aula.

---

**→ Próxima Aula — Aula 03 · 17/08** — Structured output e Pydantic v2
  
Forçar o LLM a responder com um schema garantido. CKP01 R3 entregue.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*